[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eeadata/reportnet-api-python/blob/main/examples/quickstart.ipynb) [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/eeadata/reportnet-api-python/main?labpath=examples%2Fquickstart.ipynb)

# Reportnet 3 quickstart

Read your Reportnet data into pandas in a few lines. You need a Reportnet **API key**
(Reportnet → your dataflow → *Settings* → *Generate new API key*), or a Bearer token.

In [ ]:
# Only needed if reportnet_api is not installed yet (skip if it already is):
%pip install -q "reportnet-api[pandas]"

## 1. Connect
Run the next cell, **type your key into the box that appears and press Enter** (not Shift+Enter).
`connect_interactive()` uses a hidden prompt, so the key is never shown on screen or saved in the notebook.

Alternative: set the `REPORTNET_API_KEY` environment variable before starting Jupyter and use `client = connect()`.

**Good to know:** an API key only works for the one dataflow it was created for. A *dataflow* is the whole project; a *dataset* is one dataset inside it. Don't mix up their ids.

In [ ]:
from reportnet_api.easy import connect_interactive, check_access

client = connect_interactive()
check_access(client)  # read-only: which kind of credential, is it alive

## 2. Find your dataflows and datasets

In [ ]:
from reportnet_api.easy import list_dataflows, list_datasets

dataflows = list_dataflows(client)
dataflows[["id", "name"]].head(20)

In [ ]:
DATAFLOW_ID = None  # <- put one of the ids above

datasets = list_datasets(client, DATAFLOW_ID)
datasets

## 3. Read a table into pandas

In [ ]:
from reportnet_api.easy import list_tables, read_table

DATASET_ID = None  # <- a dataset_id from the table above (NOT the dataflow id)

list_tables(client, DATASET_ID)

In [ ]:
df = read_table(client, DATASET_ID, DATAFLOW_ID)  # add table_name="..." if the dataset has several tables
df.head()

## 4. When something goes wrong
Errors are plain Python exceptions with a status code. Print `err.explain()` for the likely cause:

* `AuthenticationError` (401) – key missing or expired
* `ForbiddenError` (403) – your key/user has no access to that dataflow or feature
* `NotFoundError` (404) – wrong id
* `ValidationError` (400/422) – the request was rejected as invalid
* `LockedError` (423) – another operation (e.g. an import) holds a lock; wait and retry
* `TooManyRequestsError` (429) – slow down (`retry_after` says how long)
* `ServerError` (5xx) – a server-side failure; a dead ApiKey also shows up as a 500

An API key has **Reporter-level** access only; operations for Custodians/Stewards need a Bearer token.

## 5. Where to go next
* `reportnet_api.find("export")` searches every method and helper by keyword; `easy.guide()` prints a cheat sheet.
* [Glossary](../docs/glossary.md) (dataflow, dataset, provider, job, ...) and the [function reference](../docs/reference/index.md) (arguments, required rights, live-tested caveats).
* More notebooks: [try_it_demo](try_it_demo.ipynb) (no credential needed), [compare_countries](compare_countries.ipynb), [export_and_upload](export_and_upload.ipynb), [validation](validation.ipynb).

Everything the Reportnet API offers is also available through `client.datasets`, `client.dataset_schema`, ... for more advanced use.

In [ ]:
client.close()